# Validación psicométrica de los embedding


Sobre la matriz de embeddings de la rama visual (n × 512) del GeoVision-CLIP se aplica Análisis Factorial Exploratorio con
extracción por Componentes Principales y rotación Varimax. Se determina el número de factores $m$  a
retener vía Scree Plot y varianza acumulada (criterio: m que explique ≥ 80%). Posteriormente, se especifica
un modelo de medición confirmatorio con cuatro constructos latentes hipotetizados — Carga
Antropogénica, Estrés Vegetal, Densidad Urbana, Volatilidad Atmosférica — y evalua la bondad de ajuste
con RMSEA, CFI y SRMR, para concluir sobre la validez de constructo.

## Import y Configs

In [ ]:
%pip install factor_analyzer kneed semopy -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns

from kneed import KneeLocator
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from factor_analyzer import FactorAnalyzer, Rotator
from factor_analyzer.confirmatory_factor_analyzer import (
    ConfirmatoryFactorAnalyzer,
    ModelSpecification,
)
from factor_analyzer.factor_analyzer import (
    calculate_bartlett_sphericity,
    calculate_kmo,
)

from plotly.subplots import make_subplots
import semopy


# Carga de Embeddings

In [ ]:
# ============================================================================
# 1. CARGA DE EMBEDDINGS (reemplazar con la ruta real)
# ============================================================================
def load_embeddings(data_path=None, n_samples=2000, n_features=512, synthetic=False):
    """
    Carga los embeddings reales o genera datos sintéticos para prueba.
    
    Parámetros:
        data_path: str, ruta al archivo .npy o .csv (si synthetic=False)
        n_samples: int, número de muestras (solo para sintéticos)
        n_features: int, número de características (512 en el proyecto)
        synthetic: bool, si True genera datos con estructura factorial conocida
    
    Retorna:
        X: np.ndarray de shape (n_samples, n_features)
    """
    if synthetic:
        print("🔧 Generando datos sintéticos con 4 factores latentes (solo demostración).")
        np.random.seed(42)
        # Crear factores latentes (4 factores)
        n_factors = 4
        # Matriz de cargas: cada característica se asocia a un factor dominante
        loadings_true = np.zeros((n_features, n_factors))
        block_size = n_features // n_factors
        for i in range(n_factors):
            start = i * block_size
            end = (i + 1) * block_size if i < n_factors-1 else n_features
            loadings_true[start:end, i] = np.random.uniform(0.5, 0.9, end-start)
        # Añadir cargas cruzadas pequeñas
        loadings_true += np.random.normal(0, 0.1, (n_features, n_factors))
        # Factores latentes (normalizados)
        factors = np.random.normal(0, 1, (n_samples, n_factors))
        # Ruido
        noise = np.random.normal(0, 0.5, (n_samples, n_features))
        X = factors @ loadings_true.T + noise
        # Estandarizar
        X = StandardScaler().fit_transform(X)
        return X
    else:
        if data_path is None:
            raise ValueError("Debe proporcionar data_path para embeddings reales o usar synthetic=True.")
        if data_path.endswith('.npy'):
            X = np.load(data_path)
        elif data_path.endswith('.csv'):
            X = pd.read_csv(data_path).values
        else:
            raise ValueError("Formato no soportado. Use .npy o .csv")
        print(f"✅ Embeddings cargados: {X.shape}")
        return X

# Pruebas de Adecuación (KMO y Barlett)

In [ ]:

# ============================================================================
# 2. PRUEBAS DE ADECUACIÓN (KMO y Bartlett)
# ============================================================================
def test_adequacy(X):
    print("\n" + "="*70)
    print("PRUEBAS DE ADECUACIÓN PARA ANÁLISIS FACTORIAL")
    print("="*70)
    
    # KMO
    kmo_all, kmo_model = calculate_kmo(pd.DataFrame(X))
    print(f"\nKMO global (MSA): {kmo_model:.4f}")
    if kmo_model >= 0.9:
        print("  → Excelente")
    elif kmo_model >= 0.8:
        print("  → Meritorio")
    elif kmo_model >= 0.7:
        print("  → Mediano")
    elif kmo_model >= 0.6:
        print("  → Mediocre")
    else:
        print("  → Inaceptable (no proceder con AF)")
    
    # Bartlett
    chi2, p = calculate_bartlett_sphericity(pd.DataFrame(X))
    print(f"\nPrueba de esfericidad de Bartlett:")
    print(f"  Chi-cuadrado = {chi2:.4f}")
    print(f"  p-valor = {p:.6e}")
    if p < 0.05:
        print("  → Significativo: las variables están correlacionadas, AF procede.")
    else:
        print("  → No significativo: AF no recomendado.")
    
    return kmo_model, p

# Determinación del Número de Factores

In [ ]:
# ============================================================================
# 3. DETERMINACIÓN DEL NÚMERO DE FACTORES
# ============================================================================
def determine_n_factors(X, max_factors=20, var_threshold=0.80):
    """
    Scree plot, varianza acumulada y análisis paralelo (Horn).
    Retorna número de factores sugerido.
    """
    print("\n" + "="*70)
    print("DETERMINACIÓN DEL NÚMERO DE FACTORES")
    print("="*70)
    
    # Calcular eigenvalores usando FactorAnalyzer (sin rotación, n_factors = total)
    fa = FactorAnalyzer(n_factors=X.shape[1], rotation=None)
    fa.fit(pd.DataFrame(X))
    ev, _ = fa.get_eigenvalues()
    ev = ev[:max_factors]  # solo primeros
    var_ratio = ev / ev.sum()
    cum_var = np.cumsum(var_ratio)
    
    # Scree plot con punto de codo (KneeLocator)
    x_vals = range(1, len(ev)+1)
    kneedle = KneeLocator(x_vals, ev, curve='convex', direction='decreasing')
    elbow = kneedle.elbow if kneedle.elbow is not None else 2
    
    # Varianza acumulada
    n_by_var = np.argmax(cum_var >= var_threshold) + 1
    
    # Análisis paralelo simplificado (simulación de eigenvalores aleatorios)
    n_sim = 100
    ev_sim = []
    for _ in range(n_sim):
        X_perm = np.random.permutation(X.flatten()).reshape(X.shape)
        fa_sim = FactorAnalyzer(n_factors=X.shape[1], rotation=None)
        fa_sim.fit(pd.DataFrame(X_perm))
        ev_sim.append(fa_sim.get_eigenvalues()[0])
    ev_sim_mean = np.mean(ev_sim, axis=0)
    ev_sim_95 = np.percentile(ev_sim, 95, axis=0)
    # Número de factores donde eigenvalor real > percentil 95
    n_parallel = sum(ev[:len(ev_sim_95)] > ev_sim_95[:len(ev)])
    
    print(f"\nCriterio de Kaiser (eigenvalor > 1): {sum(ev > 1)} factores")
    print(f"Codo en Scree plot (KneeLocator): {elbow} factores")
    print(f"Varianza acumulada ≥ {var_threshold*100}%: {n_by_var} factores")
    print(f"Análisis paralelo (Horn, p95): {n_parallel} factores")
    
    # Sugerencia: usar la mediana de los criterios (o el que más se repite)
    suggested = max(2, int(np.median([elbow, n_by_var, n_parallel])))
    print(f"\n✨ Número de factores sugerido: {suggested}")


In [ ]:

    # Gráficos
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    # Scree plot
    axes[0].plot(x_vals, ev, 'bo-', markersize=6, label='Eigenvalores reales')
    axes[0].axhline(y=1, color='r', linestyle='--', label='Kaiser (λ=1)')
    axes[0].axvline(x=elbow, color='g', linestyle='--', label=f'Codo ≈ {elbow}')
    axes[0].set_title('Scree Plot')
    axes[0].set_xlabel('Número de factor')
    axes[0].set_ylabel('Eigenvalor')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    # Varianza acumulada
    axes[1].bar(x_vals, var_ratio, alpha=0.6, label='Var. individual')
    axes[1].plot(x_vals, cum_var, 'ro-', label='Var. acumulada')
    axes[1].axhline(y=var_threshold, color='orange', linestyle='--', label=f'{var_threshold*100}%')
    axes[1].set_title('Varianza explicada')
    axes[1].set_xlabel('Número de factor')
    axes[1].set_ylabel('Proporción de varianza')
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig("scree_plot_varianza.png", dpi=150)
    plt.show()
    
    return suggested


# Análisis Factorial Exploratorio (AFE)

In [ ]:

    
# ============================================================================
# 4. ANÁLISIS FACTORIAL EXPLORATORIO (EFA)
# ============================================================================
def efa_analysis(X, n_factors):
    print("\n" + "="*70)
    print(f"ANÁLISIS FACTORIAL EXPLORATORIO ({n_factors} factores, rotación Varimax)")
    print("="*70)
    
    fa = FactorAnalyzer(n_factors=n_factors, rotation='varimax', method='ml')
    fa.fit(pd.DataFrame(X))
    
    # Cargas factoriales
    loadings = fa.loadings_
    factor_names = [f'Factor{i+1}' for i in range(n_factors)]
    loadings_df = pd.DataFrame(loadings, columns=factor_names)
    print("\nMatriz de cargas factoriales rotadas (Varimax):")
    print(loadings_df.round(3).to_string())
    
    # Comunalidades
    communalities = fa.get_communalities()
    comm_df = pd.DataFrame({'Variable': [f'Dim_{i}' for i in range(X.shape[1])], 
                            'Comunalidad': communalities}).sort_values('Comunalidad', ascending=False)
    print("\nComunalidades (h²) - proporción de varianza explicada por los factores:")
    print(comm_df.head(10).to_string(index=False))
    
    # Varianza explicada por factor
    var_exp = fa.get_factor_variance()
    var_df = pd.DataFrame(var_exp, index=['SS Loadings', '% Varianza', '% Acumulado'], columns=factor_names)
    print("\nVarianza explicada por factor:")
    print(var_df.round(4).to_string())
    
    # Heatmap de cargas
    plt.figure(figsize=(12, 8))
    sns.heatmap(loadings_df, annot=True, fmt='.2f', cmap='RdYlGn', center=0, vmin=-1, vmax=1)
    plt.title(f'Cargas factoriales rotadas - {n_factors} factores')
    plt.tight_layout()
    plt.savefig("efa_loadings_heatmap.png", dpi=150)
    plt.show()
    
    # Asignación de ítems a factores (carga máxima > 0.4)
    max_load = loadings_df.abs().max(axis=1)
    assignment = loadings_df.idxmax(axis=1)
    assignment[max_load < 0.4] = 'No asignado'
    print("\nAsignación de cada dimensión al factor con carga máxima (|carga|>0.4):")
    print(assignment.value_counts())
    
    return loadings_df, fa


# 5. Análisis Factorial Confirmatorio (AFC)


In [ ]:

# ============================================================================
# 5. ANÁLISIS FACTORIAL CONFIRMATORIO (CFA)
# ============================================================================
def cfa_analysis(X, loadings_df, n_factors=4):
    """
    Especifica un modelo de medición con cuatro constructos hipotéticos:
    - CargaAntropogenica
    - EstresVegetal
    - DensidadUrbana
    - VolatilidadAtmosferica
    
    Se asigna cada variable observada (dimensión del embedding) al factor
    donde tuvo mayor carga absoluta en el EFA.
    """
    print("\n" + "="*70)
    print("ANÁLISIS FACTORIAL CONFIRMATORIO (CFA) - Modelo de 4 factores")
    print("="*70)
    
    # Obtener asignación de ítems a factores desde EFA (carga máxima > 0.4)
    # Usar los nombres teóricos
    theoretical_factors = {
        'CargaAntropogenica': [],
        'EstresVegetal': [],
        'DensidadUrbana': [],
        'VolatilidadAtmosferica': []
    }
    
    # Mapeo de nombres de factores de EFA a teóricos (podría ser directo si los ordenamos)
    # Como el orden puede variar, asignamos según el patrón de cargas (esto es un ejemplo)
    # En la práctica, el investigador debe revisar las cargas y nombrar los factores.
    # Aquí haremos una asignación automática basada en el índice del factor (1..4)
    # asumiendo que el EFA ya ordenó los factores relevantemente. Si no, se puede hacer
    # manualmente.
    
    # Simplemente para demostración, usamos los factores del EFA directamente
    # y les asignamos nombres teóricos basados en el orden.
    efa_factors = loadings_df.columns
    # Reemplazar nombres por los teóricos (puedes ajustar este mapeo)
    name_mapping = {efa_factors[0]: 'CargaAntropogenica',
                    efa_factors[1]: 'EstresVegetal',
                    efa_factors[2]: 'DensidadUrbana',
                    efa_factors[3]: 'VolatilidadAtmosferica'}
    
    # Construir diccionario de asignación
    for var_idx in range(len(loadings_df)):
        max_factor = loadings_df.iloc[var_idx].abs().idxmax()
        if max_factor in name_mapping:
            theoretical_factors[name_mapping[max_factor]].append(f'v{var_idx}')
    
    # Construir modelo semopy
    model_str = ""
    for factor, items in theoretical_factors.items():
        if items:
            model_str += f"{factor} =~ {' + '.join(items)}\n"
    print("\nModelo CFA especificado:\n", model_str)
    
    # Preparar datos: DataFrame con columnas v0, v1, ..., v_{n-1}
    df_cfa = pd.DataFrame(X, columns=[f'v{i}' for i in range(X.shape[1])])
    
    # Ajustar modelo
    try:
        model = semopy.Model(model_str)
        model.fit(df_cfa)
        # Estadísticos de ajuste
        stats = semopy.calc_stats(model)
        print("\n=== ÍNDICES DE BONDAD DE AJUSTE ===")
        print(stats.T)
        
        # Extraer índices clave
        rmsea = stats.loc['RMSEA', 'Value'] if 'RMSEA' in stats.index else None
        cfi = stats.loc['CFI', 'Value'] if 'CFI' in stats.index else None
        srmr = stats.loc['SRMR', 'Value'] if 'SRMR' in stats.index else None
        chisq = stats.loc['Chi-square', 'Value'] if 'Chi-square' in stats.index else None
        pval = stats.loc['Chi-square p-value', 'Value'] if 'Chi-square p-value' in stats.index else None
        
        print("\nEvaluación de umbrales (proyecto):")
        print(f"  RMSEA = {rmsea:.4f}  (objetivo <0.08) → {'✓ Cumple' if rmsea < 0.08 else '✗ No cumple'}")
        print(f"  CFI   = {cfi:.4f}  (objetivo >0.90) → {'✓ Cumple' if cfi > 0.90 else '✗ No cumple'}")
        print(f"  SRMR  = {srmr:.4f}  (objetivo <0.08) → {'✓ Cumple' if srmr < 0.08 else '✗ No cumple'}")
        if pval is not None:
            print(f"  χ² p-valor = {pval:.4f}  (no significativo deseable) → {'✓' if pval > 0.05 else '✗'}")
        
        # Cargas factoriales estandarizadas
        print("\nCargas factoriales estimadas (estandarizadas):")
        params = model.inspect()
        std_est = params[params['op'] == '=~'][['lval', 'rval', 'Estimate']]
        print(std_est)
        
    except Exception as e:
        print(f"Error en CFA: {e}")
        print("El modelo puede ser demasiado complejo para la matriz de datos. Considere reducir el número de variables observadas (agrupando dimensiones) o aumentar el tamaño de muestra.")
    
    return model if 'model' in locals() else None


In [ ]:
# ============================================================================
# 6. FUNCIÓN PRINCIPAL
# ============================================================================
def run_psychometric_validation(data_path=None, synthetic=True, n_samples=2000, n_features=512):
    """
    Ejecuta todo el flujo de validación psicométrica.
    
    Parámetros:
        data_path: str, ruta al archivo de embeddings (si synthetic=False)
        synthetic: bool, usar datos sintéticos (para prueba)
        n_samples: int, número de muestras (solo sintético)
        n_features: int, número de características (debe ser 512 para embeddings reales)
    """
    # Cargar embeddings
    X = load_embeddings(data_path, n_samples, n_features, synthetic)
    print(f"Datos cargados: {X.shape[0]} muestras, {X.shape[1]} dimensiones.")
    
    # 1. Pruebas de adecuación
    kmo, bartlett_p = test_adequacy(X)
    if bartlett_p > 0.05 or kmo < 0.6:
        print("\n⚠️ Advertencia: Los datos no son adecuados para análisis factorial. Continuar con precaución.")
    
    # 2. Determinar número de factores (máximo 20)
    n_factors = determine_n_factors(X, max_factors=20, var_threshold=0.80)
    
    # 3. EFA
    loadings_df, fa_model = efa_analysis(X, n_factors)
    
    # 4. CFA (solo si tenemos al menos 4 factores y suficiente muestra)
    if n_factors >= 4 and X.shape[0] > 100:
        cfa_analysis(X, loadings_df, n_factors=4)
    else:
        print("\n⏭️ No se ejecuta CFA porque el número de factores sugerido es menor a 4 o muestra insuficiente.")
    
    print("\n✅ Validación psicométrica completada.")

# ============================================================================
# EJECUCIÓN
# ============================================================================
if __name__ == "__main__":
    # Configuración:
    # - Para utilizar embeddings reales, cambie synthetic=False y proporcione data_path.
    # - Ejemplo: run_psychometric_validation(data_path="embeddings.npy", synthetic=False)
    
    # DEMOSTRACIÓN CON DATOS SINTÉTICOS
    run_psychometric_validation(synthetic=True, n_samples=2000, n_features=512)